# 15 — CPU-only retained-output analysis and corrected statistical figures


## Environment

Install only lightweight CPU packages if needed:

```bash
python -m pip install 'numpy>=1.26,<3' 'pandas>=2,<3' 'scikit-learn>=1.4,<2' 'matplotlib>=3.7,<4' 'openpyxl>=3.1,<4' jupyterlab
```

Running all cells overwrites only code-generated statistical figures in `finalized_paper_q1/figures/` and writes the new experiment tables to `finalized_paper_q1/tables/`. (`finalized_paper_q1` is the repository's existing finalized-output directory.) Diagrams are excluded by construction.


In [1]:
from pathlib import Path
import json, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, classification_report, cohen_kappa_score,
    confusion_matrix, f1_score, matthews_corrcoef, roc_auc_score)

warnings.filterwarnings('ignore', category=FutureWarning)

def locate_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'notebooks').is_dir() and (candidate / 'finalized_paper_q1').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the BanglaCyberBench repository')

ROOT = locate_root()
FINALIZED_OUTPUTS = ROOT / 'finalized_paper_q1'
FIGURE_DIR = FINALIZED_OUTPUTS / 'figures'
TABLE_DIR = FINALIZED_OUTPUTS / 'tables'
LABELS = ['abusive', 'none', 'religious', 'sexual', 'threat']
DISPLAY = ['Abusive', 'None', 'Religious', 'Sexual', 'Threat']
BOOTSTRAP_B = 2000
KAPPA_BOOTSTRAP_B = 5000
SEED = 42
OVERWRITE_PAPER_FIGURES = True

print('Project root:', ROOT)
print('Device policy: CPU only; no model loading or training')
print('Diagram policy: D1/D2/D3 are never read or written')


Project root: /Users/sefayet/Desktop/Github/BanglaCyberBench
Device policy: CPU only; no model loading or training
Diagram policy: D1/D2/D3 are never read or written


## CPU-only analyses

All results are derived from the project's retained test predictions, probabilities, split CSVs, and completed annotation workbook. The retained project data are treated as authoritative; this notebook does not compare source sizes with external metadata or audit dataset ingestion. New analysis tables go only to `finalized_paper_q1/tables/`; this cell does not change figures or diagrams.


In [2]:
started = time.time()
TABLE_DIR.mkdir(parents=True, exist_ok=True)

def read_csv(path, **kwargs):
    return pd.read_csv(path, keep_default_na=False, **kwargs)

def expected_calibration_error(y, proba, bins=15):
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    edges = np.linspace(0, 1, bins + 1)
    value = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi) if lo else (conf >= lo) & (conf <= hi)
        if mask.any():
            value += mask.mean() * abs((pred[mask] == y[mask]).mean() - conf[mask].mean())
    return float(value)

def multiclass_brier(y, proba):
    onehot = np.eye(proba.shape[1])[y]
    return float(np.mean(np.sum((proba - onehot) ** 2, axis=1)))

def supported_macro_auroc(y, proba):
    present = np.unique(y)
    if len(present) < 2:
        return np.nan
    onehot = np.eye(len(LABELS))[y]
    scores = []
    for c in present:
        target = onehot[:, c]
        if target.min() != target.max():
            scores.append(roc_auc_score(target, proba[:, c]))
    return float(np.mean(scores)) if scores else np.nan

def metric_row(name, y, pred, proba):
    present = np.unique(y)
    return {
        'Subset': name, 'n': len(y),
        'Macro-F1 (5 classes)': f1_score(y, pred, labels=np.arange(5), average='macro', zero_division=0),
        'Macro-F1 (supported classes)': f1_score(y, pred, labels=present, average='macro', zero_division=0),
        'Weighted-F1': f1_score(y, pred, average='weighted', zero_division=0),
        'Accuracy': accuracy_score(y, pred), 'MCC': matthews_corrcoef(y, pred),
        'Macro-AUROC (supported)': supported_macro_auroc(y, proba),
        'ECE-15': expected_calibration_error(y, proba), 'Brier': multiclass_brier(y, proba),
    }

test = read_csv(ROOT / 'data/splits/random_test.csv')
pred_path = ROOT / 'outputs/ensemble/test_pred.npy'
proba_path = ROOT / 'outputs/ensemble/test_proba.npy'
pred = np.load(pred_path).astype(int)
proba = np.load(proba_path).astype(float)
encoder = json.loads((ROOT / 'outputs/models_main/label_encoder.json').read_text())
label_to_id = {str(k): int(v) for k, v in encoder.items()}
assert set(label_to_id) == set(LABELS), label_to_id
y = test['label5'].map(label_to_id).to_numpy(dtype=int)
assert len(test) == len(pred) == len(proba), (len(test), len(pred), len(proba))
assert proba.shape == (len(test), 5)
assert np.allclose(proba.sum(axis=1), 1.0, atol=1e-4)

rows = [metric_row('All', y, pred, proba)]
for script in ['bangla', 'romanized']:
    m = test['script'].str.lower().eq(script).to_numpy()
    rows.append(metric_row(script.title(), y[m], pred[m], proba[m]))
main_by_script = pd.DataFrame(rows)

support_by_script = (test.groupby(['script', 'label5']).size().unstack(fill_value=0)
                     .reindex(columns=LABELS, fill_value=0).reset_index())

per_class_rows = []
for subset, mask in [('All', np.ones(len(test), dtype=bool)),
                     ('Bangla', test['script'].str.lower().eq('bangla').to_numpy()),
                     ('Romanized', test['script'].str.lower().eq('romanized').to_numpy())]:
    report = classification_report(y[mask], pred[mask], labels=np.arange(5),
                                   target_names=DISPLAY, output_dict=True, zero_division=0)
    for label in DISPLAY:
        per_class_rows.append({'Subset': subset, 'Class': label, **report[label]})
per_class_by_script = pd.DataFrame(per_class_rows)

def bootstrap_headline(y, pred, B=BOOTSTRAP_B, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    values = np.empty((B, 4), dtype=float)
    for b in range(B):
        idx = rng.integers(0, n, n)
        yt, yp = y[idx], pred[idx]
        values[b] = [f1_score(yt, yp, average='macro', zero_division=0),
                     f1_score(yt, yp, average='weighted', zero_division=0),
                     accuracy_score(yt, yp), matthews_corrcoef(yt, yp)]
    names = ['Macro-F1', 'Weighted-F1', 'Accuracy', 'MCC']
    point = values.mean(axis=0)
    lo, hi = np.percentile(values, [2.5, 97.5], axis=0)
    return pd.DataFrame({'Metric': names, 'Bootstrap mean': point, 'CI low': lo, 'CI high': hi})

bootstrap_overall = bootstrap_headline(y, pred)

annotation = pd.read_excel(ROOT / 'outputs/annotation/annotation_sample_BLANK.xlsx')
a = annotation['annotator_1_label'].astype(str).str.strip().str.lower()
b = annotation['annotator_2_label'].astype(str).str.strip().str.lower()
valid = a.isin(LABELS) & b.isin(LABELS)
a, b = a[valid].to_numpy(), b[valid].to_numpy()
kappa = cohen_kappa_score(a, b, labels=LABELS)
rng = np.random.default_rng(SEED)
kappa_boot = np.empty(KAPPA_BOOTSTRAP_B)
for i in range(KAPPA_BOOTSTRAP_B):
    idx = rng.integers(0, len(a), len(a))
    kappa_boot[i] = cohen_kappa_score(a[idx], b[idx], labels=LABELS)
kappa_ci = pd.DataFrame([{'n': len(a), 'Agreement': float(np.mean(a == b)), 'Kappa': kappa,
                          'CI low': np.nanpercentile(kappa_boot, 2.5),
                          'CI high': np.nanpercentile(kappa_boot, 97.5),
                          'Bootstrap B': KAPPA_BOOTSTRAP_B}])

holdout_rows, prior_rows = [], []
holdout_dir = ROOT / 'data/splits/source_holdout_bangla_only'
for source in ['facebook_44001', 'multilabel_12557', 'bd_shs']:
    parts = {}
    for split in ['train', 'val', 'test']:
        frame = read_csv(holdout_dir / f'source_holdout_{source}_{split}.csv')
        parts[split] = frame
    holdout_rows.append({'Held-out source': source, **{f'{k}_n': len(v) for k, v in parts.items()}})
    for split, frame in parts.items():
        counts = frame['label5'].value_counts()
        for label in LABELS:
            prior_rows.append({'Held-out source': source, 'Split': split, 'Class': label,
                               'n': int(counts.get(label, 0)), 'Share': float((frame['label5'] == label).mean())})
holdout_sizes = pd.DataFrame(holdout_rows)
holdout_priors = pd.DataFrame(prior_rows)

tables = {
 'main_results_by_script': main_by_script, 'class_by_script_support': support_by_script,
 'main_per_class_by_script': per_class_by_script, 'main_bootstrap_ci': bootstrap_overall,
 'iaa_kappa_bootstrap': kappa_ci,
 'source_holdout_sizes': holdout_sizes, 'source_holdout_priors': holdout_priors,
}
for name, frame in tables.items():
    frame.to_csv(TABLE_DIR / f'table_cpu_{name}.csv', index=False)

display(main_by_script.round(4))
display(support_by_script)
display(kappa_ci.round(4))


,Subset,n,Macro-F1 (5 classes),Macro-F1 (supported classes),Weighted-F1,Accuracy,MCC,Macro-AUROC (supported),ECE-15,Brier
0,All,18865,0.8225,0.8225,0.8332,0.8339,0.7452,0.9626,0.0328,0.2482
1,Bangla,11406,0.8303,0.8303,0.8368,0.8369,0.7805,0.9671,0.0348,0.2454
2,Romanized,7459,0.3900,0.4875,0.8238,0.8293,0.5771,0.8743,0.0301,0.2524


label5,script,abusive,none,religious,sexual,threat
0,bangla,2916,4162,1576,2114,638
1,romanized,2077,5301,30,51,0


,n,Agreement,Kappa,CI low,CI high,Bootstrap B
0,500,0.8,0.7094,0.6583,0.7579,5000


## Regenerate the statistical figures in place

This cell deliberately uses no in-plot title/subtitle; the paper caption carries that information. Labels are horizontal or given enough margin, the two confusion panels use identical canvas geometry, and every figure is exported as PNG plus vector PDF/SVG. The historical filenames are retained so LaTeX references do not change. No diagram path appears anywhere in this cell.


In [3]:
COL = {'blue':'#377EB8', 'sky':'#56B4E9', 'green':'#009E73', 'orange':'#E69F00',
       'red':'#D55E00', 'purple':'#7B61A8', 'gray':'#7A7A7A', 'ink':'#1F2937', 'grid':'#D1D5DB'}
plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':9.5, 'axes.labelsize':10,
                     'xtick.labelsize':9, 'ytick.labelsize':9, 'legend.fontsize':8.5,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'pdf.fonttype':42, 'ps.fonttype':42, 'svg.fonttype':'none'})
generated_figures = []

def clean(ax, axis='y'):
    ax.grid(True, axis=axis, color=COL['grid'], linewidth=.55, alpha=.65)
    ax.set_axisbelow(True)
    return ax

def save_figure(fig, name):
    assert not name.lower().startswith(('d1_', 'd2_', 'd3_')), 'Diagram writes are forbidden'
    if not OVERWRITE_PAPER_FIGURES:
        plt.close(fig); return
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    vector_dir = FIGURE_DIR / 'other_formats'
    vector_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / f'{name}.png', dpi=600, bbox_inches='tight', pad_inches=.05, facecolor='white')
    fig.savefig(vector_dir / f'{name}.pdf', bbox_inches='tight', pad_inches=.05, facecolor='white')
    fig.savefig(vector_dir / f'{name}.svg', bbox_inches='tight', pad_inches=.05, facecolor='white')
    generated_figures.append(name)
    plt.close(fig)

# Composition: large donut charts retain the intended pie-chart design.
# Names/counts are placed in external legends and percentages inside the wedges.
benchmark = read_csv(ROOT / 'data/processed/benchmark_cleaned.csv')
composition = [
    ('Source', benchmark['source'].value_counts(), [COL['blue'], COL['green'], COL['orange'], COL['purple']]),
    ('Script', benchmark['script'].replace({'bangla':'Bangla', 'romanized':'Romanized'}).value_counts(), [COL['blue'], COL['orange']]),
    ('Class', benchmark['label5'].str.title().value_counts(), [COL['green'], COL['blue'], COL['orange'], COL['purple'], COL['red']]),
]
pretty_group = {
    'facebook_44001': 'Facebook-44K', 'banth': 'BanTH',
    'multilabel_12557': 'Multilabel-12.5K', 'bd_shs': 'BD-SHS',
}
fig, axes = plt.subplots(1, 3, figsize=(15.6, 6.3), constrained_layout=True)
for ax, (group, series, colors) in zip(axes, composition):
    series = series.sort_values(ascending=False)
    wedges, _, _ = ax.pie(
        series.values, labels=None, colors=colors[:len(series)], startangle=90,
        counterclock=False, autopct=lambda pct: f'{pct:.1f}%', radius=1.10,
        pctdistance=.77, textprops={'fontsize': 9.5, 'fontweight': 'bold'},
        wedgeprops={'width': .48, 'edgecolor': 'white', 'linewidth': 1.2},
    )
    total = series.sum()
    legend_labels = [
        f'{pretty_group.get(str(name), str(name))}: {value:,} ({100*value/total:.1f}%)'
        for name, value in series.items()
    ]
    ax.legend(wedges, legend_labels, loc='upper center', bbox_to_anchor=(.5, -.08),
              frameon=False, fontsize=8.3, ncol=1, handlelength=1.1, handletextpad=.5)
    ax.text(0, 0, group, ha='center', va='center', fontsize=10.5,
            fontweight='bold', color=COL['ink'])
    ax.set_aspect('equal')
save_figure(fig, 'fig_dataset_composition_pies')

# Main results.
main = read_csv(TABLE_DIR / 'table1_main_results.csv')
metrics = ['Macro-F1', 'Weighted-F1', 'Accuracy', 'MCC', 'Macro-AUROC']
fig, ax = plt.subplots(figsize=(7.8, 5.2), constrained_layout=True)
x, w = np.arange(len(metrics)), .36
for i, (_, row) in enumerate(main.iterrows()):
    bars = ax.bar(x + (i-.5)*w, row[metrics].astype(float), w,
                  label=row['System'], color=[COL['gray'], COL['blue']][i])
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=7.5)
ax.set_xticks(x, metrics, rotation=18, ha='right'); ax.set_ylabel('Score'); ax.set_ylim(.60, 1.01)
ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(.5, 1.02), ncol=2,
          borderaxespad=0, columnspacing=1.5, handletextpad=.6)
clean(ax)
save_figure(fig, 'fig02_main_results')

# Per-class F1.
full_class = per_class_by_script.query("Subset == 'All'").copy().sort_values('f1-score')
fig, ax = plt.subplots(figsize=(6.4, 4.2))
bars = ax.barh(full_class['Class'], full_class['f1-score'], color=COL['blue'])
ax.bar_label(bars, fmt='%.3f', padding=3); ax.set_xlabel('F1'); ax.set_xlim(0, 1); clean(ax, 'x')
save_figure(fig, 'fig03_per_class_f1')

# Confusion panels: deliberately identical geometry.
cm = confusion_matrix(y, pred, labels=np.arange(5))
fig, ax = plt.subplots(figsize=(6.4, 5.8), constrained_layout=True)
im = ax.imshow(cm, cmap='Blues')
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center', fontsize=8,
                color='white' if cm[i,j] > cm.max()/2 else COL['ink'])
ax.set_xticks(range(5), DISPLAY, rotation=25, ha='right'); ax.set_yticks(range(5), DISPLAY)
ax.set_xlabel('Predicted class'); ax.set_ylabel('True class'); fig.colorbar(im, ax=ax, shrink=.78)
save_figure(fig, 'fig04_confusion_matrix')

pairs = []
for i in range(5):
    for j in range(5):
        if i != j: pairs.append((f'{DISPLAY[i]} → {DISPLAY[j]}', int(cm[i,j])))
pairs = sorted(pairs, key=lambda z: z[1], reverse=True)[:8][::-1]
fig, ax = plt.subplots(figsize=(6.4, 5.8), constrained_layout=True)
bars = ax.barh([p[0] for p in pairs], [p[1] for p in pairs], color=COL['red'], alpha=.88)
ax.bar_label(bars, padding=3); ax.set_xlabel('Misclassified comments'); clean(ax, 'x')
save_figure(fig, 'fig05_top_confusions')

# Component and taxonomy sensitivity: no title/subtitle collision.
abl = read_csv(ROOT / 'outputs/ablation/component_ablation.csv')
fig, ax = plt.subplots(figsize=(7.5, 4.6), constrained_layout=True)
x = np.arange(len(abl)); bars = ax.bar(x, abl['macro_f1_mean'], yerr=abl['macro_f1_std'],
                                       capsize=3, color=COL['blue'])
ax.bar_label(bars, labels=[f'{v:.3f}' for v in abl['macro_f1_mean']], padding=4, fontsize=7.5)
ax.set_xticks(x, abl['config'], rotation=28, ha='right'); ax.set_ylabel('Macro-F1'); ax.set_ylim(.77, .83); clean(ax)
save_figure(fig, 'fig06_component_ablation')

tax = read_csv(ROOT / 'outputs/ablation/taxonomy_ablation.csv')
tax_value_col = 'macro_f1_mean' if 'macro_f1_mean' in tax.columns else 'macro_f1'
tax_error = tax['macro_f1_std'] if 'macro_f1_std' in tax.columns else None
fig, ax = plt.subplots(figsize=(5.2, 4.2), constrained_layout=True)
bars = ax.bar(tax['config'], tax[tax_value_col], yerr=tax_error, capsize=4,
              color=[COL['blue'], COL['orange']])
ax.bar_label(bars, labels=[f'{v:.3f}' for v in tax[tax_value_col]], padding=4)
ax.set_ylabel('Macro-F1'); ax.set_ylim(0, .9); clean(ax)
save_figure(fig, 'fig07_taxonomy_ablation')

# Source robustness with readable labels.
rob = read_csv(ROOT / 'outputs/robustness/robustness_summary.csv').sort_values('macro_f1_mean')
pretty_source = {'facebook_44001':'Facebook-44K', 'multilabel_12557':'Multilabel-12.5K', 'bd_shs':'BD-SHS'}
fig, ax = plt.subplots(figsize=(6.8, 4.4), constrained_layout=True)
bars = ax.barh(rob['held_out'].map(pretty_source), rob['macro_f1_mean'], xerr=rob['macro_f1_std'],
               capsize=3, color=COL['purple'])
ax.axvline(main_by_script.iloc[0]['Macro-F1 (5 classes)'], color=COL['red'], ls='--', label='In-domain')
ax.bar_label(bars, labels=[f'{v:.3f}' for v in rob['macro_f1_mean']], padding=5)
ax.set_xlabel('Macro-F1'); ax.set_xlim(0, .9); ax.legend(frameon=False); clean(ax, 'x')
save_figure(fig, 'fig08_robustness')

# Base-paper figure: Threat F1 is recomputed from reported precision/recall, not copied from recall.
comparison = json.loads((ROOT / 'outputs/basepaper/comparison.json').read_text())
base = comparison['base_paper']['per_class'].copy()
base['Threat'] = 2 * 0.8924 * 0.7579 / (0.8924 + 0.7579)
classes = ['Not Bully', 'Religious', 'Sexual', 'Threat', 'Troll']
ours = comparison['ours']['per_class_f1']
fig, ax = plt.subplots(figsize=(7.8, 5.1), constrained_layout=True)
x, w = np.arange(5), .36
b1 = ax.bar(x-w/2, [base[c] for c in classes], w, color=COL['gray'], label='Base paper')
b2 = ax.bar(x+w/2, [ours[c] for c in classes], w, color=COL['blue'], label='Proposed model')
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - .006,
                f'{bar.get_height():.3f}', ha='center', va='top', fontsize=7.5,
                color='white', fontweight='bold')
ax.set_xticks(x, classes, rotation=18, ha='right'); ax.set_ylabel('F1'); ax.set_ylim(.72, .98)
ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(.5, 1.02), ncol=2,
          borderaxespad=0)
clean(ax)
save_figure(fig, 'fig09_basepaper_comparison')

weights = read_csv(TABLE_DIR / 'table7_ensemble_weights.csv').sort_values('Weight')
fig, ax = plt.subplots(figsize=(7.2, 5.0), constrained_layout=True)
bars = ax.barh(weights['Run'], weights['Weight'], color=COL['green'])
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=7.5); ax.set_xlabel('Fusion weight'); clean(ax, 'x')
save_figure(fig, 'fig10_ensemble_weights')

ci = bootstrap_overall
fig, ax = plt.subplots(figsize=(6.4, 4.5), constrained_layout=True)
x = np.arange(len(ci)); values = ci['Bootstrap mean'].to_numpy()
err = np.vstack([values-ci['CI low'].to_numpy(), ci['CI high'].to_numpy()-values])
ax.errorbar(x, values, yerr=err, fmt='o', color=COL['blue'], ecolor=COL['ink'], capsize=5, ms=7)
ax.set_xticks(x, ci['Metric'], rotation=15, ha='right'); ax.set_ylabel('Score'); ax.set_ylim(.70, .87); clean(ax)
save_figure(fig, 'fig11_bootstrap_ci')

print('Generated/overwritten statistical figures:', generated_figures)
print('Confirmed: no diagram output path was used.')


Generated/overwritten statistical figures: ['fig_dataset_composition_pies', 'fig02_main_results', 'fig03_per_class_f1', 'fig04_confusion_matrix', 'fig05_top_confusions', 'fig06_component_ablation', 'fig07_taxonomy_ablation', 'fig08_robustness', 'fig09_basepaper_comparison', 'fig10_ensemble_weights', 'fig11_bootstrap_ci']
Confirmed: no diagram output path was used.


## Save a compact run manifest

After you run the notebook, send `finalized_paper_q1/tables/table_cpu_run_manifest.json` back to Codex. The requested Markdown handoff can then report the new results, explain them, and give exact manuscript replacements.


In [4]:
manifest = {
 'status': 'completed', 'runtime_seconds': round(time.time() - started, 2),
 'device': 'CPU only', 'bootstrap_B': BOOTSTRAP_B, 'kappa_bootstrap_B': KAPPA_BOOTSTRAP_B,
 'generated_figures': generated_figures, 'diagrams_touched': [],
 'analysis_tables': sorted(tables),
}
(TABLE_DIR / 'table_cpu_run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
assert manifest['diagrams_touched'] == []


{
  "status": "completed",
  "runtime_seconds": 26.15,
  "device": "CPU only",
  "bootstrap_B": 2000,
  "kappa_bootstrap_B": 5000,
  "generated_figures": [
    "fig_dataset_composition_pies",
    "fig02_main_results",
    "fig03_per_class_f1",
    "fig04_confusion_matrix",
    "fig05_top_confusions",
    "fig06_component_ablation",
    "fig07_taxonomy_ablation",
    "fig08_robustness",
    "fig09_basepaper_comparison",
    "fig10_ensemble_weights",
    "fig11_bootstrap_ci"
  ],
  "diagrams_touched": [],
  "analysis_tables": [
    "class_by_script_support",
    "iaa_kappa_bootstrap",
    "main_bootstrap_ci",
    "main_per_class_by_script",
    "main_results_by_script",
    "source_holdout_priors",
    "source_holdout_sizes"
  ]
}
